# Surface Cutting Example
This notebook generates slab structures, applies constraints, assigns charges, and writes a CIF file.

In [2]:
from ase.io import read
from ase.io import write
from ase.visualize import view
from ase.constraints import FixAtoms
import sys

sys.path.append("..") # add parent directory to path to import helper functions
from helper_functions import generate_pymatgen_surface as surface


Read the bulk structure.

In [3]:
bulk_model = read("data/TiNiO3.cif")

# Inspect the bulk structure via GUI (optional)
# view(bulk_model)


Generate surfaces with a vacuum region.

In [4]:
vacuum = 20  # in Angstrom
slabs = surface(
    bulk_model,
    layers=1,  # use 2 to see more terminations
    symmetric=True,
    miller_index=(0, 0, 1),
    vacuum=vacuum,
    spin=True,
    tol=0.01,
)


c:\Users\c1528354\GitHub\chem4energy-2026\chem4energy-2026\.venv\Lib\site-packages\pymatgen\core\surface.py:1361: UserWarning: Too many sites removed, please use a larger slab.
  sym_slabs = self.nonstoichiometric_symmetrized_slab(group[0])
c:\Users\c1528354\GitHub\chem4energy-2026\chem4energy-2026\.venv\Lib\site-packages\pymatgen\core\surface.py:1361: UserWarning: Too many sites removed, please use a larger slab.
  sym_slabs = self.nonstoichiometric_symmetrized_slab(group[0])


Constrain the bottom 6 Angstroms of each slab.

In [5]:
for slab in slabs:
    c = FixAtoms(mask=[atom.index for atom in slab if atom.z < vacuum + 6])
    slab.set_constraint(c)


Adjust initial charges by species.

In [6]:
species_charges = {"Ti": 4, "Ni": 2, "O": -2}
for slab in slabs:
    for atom in slab:
        atom.charge = species_charges.get(atom.symbol)


Optional: assign a pre-supplied list of charges.

```python
list_of_charges = []
slab.set_initial_charges(list_of_charges)
```


Select one slab, build a supercell, and save as CIF.

In [ ]:
atoms = slabs[0]
atoms = atoms.repeat((2, 2, 1))

# view(atoms)

write("data/TiNiO3_slab.cif", atoms)
